## S&P 500 RSI Strength + 52-Week High Scanner

In [ ]:
# ============================================================
# S&P 500 RSI STRENGTH + 52-WEEK HIGH SCANNER
#
# Can run standalone OR after the S&P 500 Stock Scanner cell.
# If stock_data / spy_close are already in scope (from the
# scanner cell), they are reused — no re-download needed.
# Otherwise this cell downloads everything itself.
#
# Filters (ALL three must be true):
#   1. RSI(14) of stock >= RSI(14) of SPY + 5 pts
#      (stock is meaningfully stronger than the benchmark)
#   2. Latest close is within 2% of its 52-week high
#      (stock is at or making a new 52-week high)
#   3. Latest close is within ±10% of its 50-day SMA
#      (price is not excessively extended from the mean)
#
# Output:
#   /content/sp500_rsi_scan.csv
#   /content/sp500_rsi_watchlist.txt
# ============================================================

import csv as _csv, io, time, warnings
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

OUTPUT_RSI_CSV       = "/content/sp500_rsi_scan.csv"
OUTPUT_RSI_WATCHLIST = "/content/sp500_rsi_watchlist.txt"
BATCH_SIZE           = 50

# ── Parameters ───────────────────────────────────────────────
RSI_PERIOD      = 14
RSI_MIN_EDGE    = 5.0   # stock RSI must beat SPY RSI by at least this many points
HIGH_52W_BUFFER = 0.02  # within 2% of 52-week high counts as "at the high"
SMA50_BAND      = 0.10  # price within ±10% of 50-day SMA


# ── RSI (Wilder smoothing) ───────────────────────────────────
def _rsi(series, length=14):
    delta    = series.diff()
    gain     = delta.clip(lower=0)
    loss     = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=length - 1, min_periods=length).mean()
    avg_loss = loss.ewm(com=length - 1, min_periods=length).mean()
    rs       = avg_gain / avg_loss.replace(0, float("nan"))
    return 100 - (100 / (1 + rs))


print("=" * 70)
print("  S&P 500 RSI STRENGTH + 52-WEEK HIGH SCANNER")
print("=" * 70)


# ── Reuse data from scanner cell, or download fresh ──────────
try:
    _ = stock_data, spy_close, ticker_info
    print("\n  Reusing data from S&P 500 scanner cell.")
except NameError:
    print("\n  stock_data not found — downloading S&P 500 data now...")

    end_date   = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")

    # Ticker list from Wikipedia
    html  = requests.get(
        "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
        headers={"User-Agent": "Mozilla/5.0"},
    ).text
    sp500 = pd.read_html(io.StringIO(html), flavor="lxml")[0]
    sp500["yf_ticker"] = sp500["Symbol"].str.replace(".", "-", regex=False)
    ticker_info = {
        row["yf_ticker"]: {"symbol": row["Symbol"], "name": row["Security"], "sector": row["GICS Sector"]}
        for _, row in sp500.iterrows()
    }
    print(f"  Found {len(ticker_info)} tickers")

    # SPY benchmark
    spy_raw   = yf.download("SPY", start=start_date, end=end_date, auto_adjust=True, progress=False)
    spy_close = spy_raw["Close"].squeeze()

    # Stock price data
    all_tickers = list(ticker_info.keys())
    batches     = [all_tickers[i:i+BATCH_SIZE] for i in range(0, len(all_tickers), BATCH_SIZE)]
    stock_data  = {}
    for i, batch in enumerate(batches):
        print(f"  Batch {i+1}/{len(batches)}...", end=" ", flush=True)
        try:
            raw = yf.download(batch, start=start_date, end=end_date,
                              auto_adjust=True, progress=False, threads=True)
            if len(batch) == 1:
                if not raw.empty:
                    stock_data[batch[0]] = raw.rename(columns=str.lower)
            else:
                for t in batch:
                    try:
                        df = raw.xs(t, level=1, axis=1).dropna(how="all")
                        if not df.empty:
                            stock_data[t] = df.rename(columns=str.lower)
                    except KeyError:
                        pass
            print(f"ok ({sum(1 for t in batch if t in stock_data)}/{len(batch)})")
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(0.3)
    print(f"  Downloaded {len(stock_data)} stocks")


# ── SPY RSI ──────────────────────────────────────────────────
spy_rsi_ser = _rsi(pd.Series(spy_close.values, index=spy_close.index), RSI_PERIOD)
spy_rsi     = float(spy_rsi_ser.iloc[-1])
print(f"\n  SPY RSI(14)       : {spy_rsi:.2f}")
print(f"  Min stock RSI     : {spy_rsi + RSI_MIN_EDGE:.2f}  (SPY + {RSI_MIN_EDGE} pts)")
print(f"  52w-high buffer   : within {HIGH_52W_BUFFER*100:.0f}%")
print(f"  SMA50 band        : within ±{SMA50_BAND*100:.0f}%")
print(f"\n  Scanning {len(stock_data)} stocks...")


# ── Scan ─────────────────────────────────────────────────────
rsi_results = []

for yf_t, df in stock_data.items():
    if len(df) < max(RSI_PERIOD + 5, 50):
        continue

    info       = ticker_info.get(yf_t, {})
    close      = df["close"]
    last_close = float(close.iloc[-1])

    # RSI
    stock_rsi = float(_rsi(close, RSI_PERIOD).iloc[-1])
    rsi_edge  = stock_rsi - spy_rsi

    # 52-week high (full downloaded series ≈ 1 year)
    high_52w      = float(close.max())
    pct_from_high = (high_52w - last_close) / high_52w   # 0.0 = exactly at the high

    # 50-day SMA
    sma50_val = close.rolling(50, min_periods=50).mean().iloc[-1]
    if pd.isna(sma50_val):
        pct_from_sma50 = float("nan")
    else:
        pct_from_sma50 = abs(last_close - float(sma50_val)) / float(sma50_val)

    # Filters
    f_rsi      = rsi_edge >= RSI_MIN_EDGE
    f_52w_high = pct_from_high <= HIGH_52W_BUFFER
    f_sma50    = (not np.isnan(pct_from_sma50)) and (pct_from_sma50 <= SMA50_BAND)
    confirmed  = f_rsi and f_52w_high and f_sma50

    rsi_results.append({
        "symbol":            info.get("symbol", yf_t),
        "name":              info.get("name", ""),
        "sector":            info.get("sector", ""),
        "confirmed":         confirmed,
        "rsi_14":            round(stock_rsi, 2),
        "spy_rsi_14":        round(spy_rsi, 2),
        "rsi_edge":          round(rsi_edge, 2),
        "pct_from_52w_high": round(pct_from_high * 100, 2),
        "pct_from_sma50":    round(pct_from_sma50 * 100, 2) if not np.isnan(pct_from_sma50) else None,
        "last_close":        round(last_close, 4),
        "f_rsi":             f_rsi,
        "f_52w_high":        f_52w_high,
        "f_sma50":           f_sma50,
    })

confirmed_rsi = [r for r in rsi_results if r["confirmed"]]
print(f"  Scan complete : {len(rsi_results)} stocks | {len(confirmed_rsi)} passed all filters\n")


# ── Display results ──────────────────────────────────────────
print("=" * 100)
print(f"  RSI STRENGTH + 52-WEEK HIGH — CONFIRMED SIGNALS ({len(confirmed_rsi)} stocks)")
print("=" * 100)

if confirmed_rsi:
    print(f"  {'Symbol':<8} {'Name':<28} {'Sector':<24} "
          f"{'RSI':>6} {'Edge':>6} {'%Hi':>7} {'%SMA50':>8}")
    print(f"  {'-'*8} {'-'*28} {'-'*24} {'-'*6} {'-'*6} {'-'*7} {'-'*8}")
    for r in sorted(confirmed_rsi, key=lambda x: -x["rsi_edge"]):
        sma_str = f"{r['pct_from_sma50']:6.1f}%" if r["pct_from_sma50"] is not None else "   N/A"
        print(f"  {r['symbol']:<8} {r['name'][:28]:<28} {r['sector'][:24]:<24} "
              f"{r['rsi_14']:>6.1f} {r['rsi_edge']:>+6.1f} "
              f"{r['pct_from_52w_high']:>6.1f}% {sma_str:>8}")
else:
    print("  No stocks passed all three filters today.")

print("=" * 100)
print(f"\n  Legend — RSI: stock RSI(14)  |  Edge: stock RSI minus SPY RSI ({spy_rsi:.1f})")
print(f"           %Hi: % below 52-week high (0.0% = exactly at the high)")
print(f"           %SMA50: % distance from 50-day SMA")


# ── Export ───────────────────────────────────────────────────
fieldnames = [
    "symbol", "name", "sector", "confirmed",
    "rsi_14", "spy_rsi_14", "rsi_edge",
    "pct_from_52w_high", "pct_from_sma50",
    "last_close", "f_rsi", "f_52w_high", "f_sma50",
]

with open(OUTPUT_RSI_CSV, "w", newline="", encoding="utf-8") as f:
    writer = _csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(sorted(rsi_results, key=lambda r: r["confirmed"], reverse=True))

with open(OUTPUT_RSI_WATCHLIST, "w", encoding="utf-8") as f:
    for r in confirmed_rsi:
        f.write(f"{r['symbol']}\n")

print(f"\n  Full results  → {OUTPUT_RSI_CSV}")
print(f"  TV watchlist  → {OUTPUT_RSI_WATCHLIST}  ({len(confirmed_rsi)} tickers)")
print("\nImport into TradingView:")
print("  Watchlist → ⋮ → Import watchlist → sp500_rsi_watchlist.txt")